# Daily LinkedIn Post Agent — Claude Only + Python PNG Visuals

**What this notebook does:**
1. Pulls fresh tech/AI news from Hacker News + arXiv
2. Picks a daily topic from a rotating bank
3. Uses Claude to draft a LinkedIn post in your voice
4. Humanises the post by cleaning special characters and AI-looking formatting
5. Creates professional PNG visuals using Python code only, with no Gemini, Imagen, OpenAI, Pexels, or external image-generation API
6. Emails the draft and coded PNG visual to you for review

**Important visual rule:**
This version creates professional graphics such as dashboards, QA workflows, API testing reports, code snippet cards, job-search trackers, and learning roadmaps. It does **not** create realistic AI photos, quote cards, word-clouds, or random text posters.

**Best for Mohith:** Data Science, QA Automation, SDET, Python, API testing, UK job search, and learning journey posts.


## 1️⃣ Install dependencies

In [ ]:
!pip install -q anthropic feedparser requests matplotlib pillow


## 2️⃣ Configure secrets

Use Colab's **Secrets** tab, the key icon in the left sidebar. Never hardcode private keys.

Add these secrets:

| Secret name | What it is |
|---|---|
| `ANTHROPIC_API_KEY` | Your Claude API key from console.anthropic.com. Used for writing the LinkedIn post content. |
| `GMAIL_ADDRESS` | The Gmail you'll send FROM and TO |
| `GMAIL_APP_PASSWORD` | A 16-character Gmail app password |

No Gemini, Imagen, OpenAI, Pexels, or Unsplash API key is required in this version.


In [ ]:
from google.colab import userdata
import os

# Claude is used for writing the LinkedIn post content.
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

GMAIL_ADDRESS = userdata.get('GMAIL_ADDRESS')
GMAIL_APP_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')

# Where to send drafts. Change this if you want a different review inbox.
RECIPIENT_EMAIL = GMAIL_ADDRESS

if not os.environ.get('ANTHROPIC_API_KEY'):
    raise ValueError('Missing ANTHROPIC_API_KEY. Add it in Colab Secrets.')

if not GMAIL_ADDRESS or not GMAIL_APP_PASSWORD:
    print('Gmail secrets missing. The notebook can still create the post and PNG, but email sending will fail until you add Gmail secrets.')


## 3️⃣ Personalize your voice & topics

**Edit this cell to make the posts sound like YOU.** The more specific, the better.

In [ ]:
# === YOUR PROFILE ===
YOUR_NAME = "Mohith Sangeetham"
YOUR_ROLE = "MSc Data Science graduate and QA Automation / SDET professional"
YOUR_AUDIENCE = "UK recruiters, data professionals, QA/SDET engineers, AI learners, and tech hiring managers"

# === YOUR VOICE ===
# Keep this natural because the final text will also pass through the humanise_text() cleanup function.
YOUR_TONE = """
- Conversational, clear, and human-written
- Practical insights over hype
- First-person perspective where suitable
- Short paragraphs, simple wording, and natural flow
- Avoid corporate buzzwords, over-polished AI wording, heavy emojis, and hashtag stuffing
- No em dash, no fancy symbols, no unnecessary special characters
- Make it sound like a real professional sharing a useful thought, not a sales post
"""

# === HUMANISER SETTINGS ===
REMOVE_EMOJIS = True
MAX_HASHTAGS = 5

# === VISUAL CONTENT SETTINGS ===
# Claude creates the LinkedIn post text.
# Python creates the PNG visual using code only.
# No Gemini, Imagen, OpenAI, Pexels, or image-generation API is used.
USE_CODED_PNG_VISUALS = True
ATTACH_TEXT_POST_CARD = False  # Keep False. This avoids plain quote/text-card images.

# Optional: if you put real screenshots here, the notebook can use them for project posts.
PROJECT_SCREENSHOT_DIR = "/content/drive/MyDrive/linkedin_agent/project_screenshots"

# PNG designs that can be generated by code.
CODED_VISUAL_TYPES = [
    "data_dashboard",
    "qa_workflow",
    "api_testing_report",
    "python_code_snippet",
    "job_search_tracker",
    "learning_roadmap",
    "project_showcase",
    "achievement_tracker",
]

# === DAILY TOPIC BANK ===
DAILY_TOPIC_BANK = [
    "Data Science learning journey",
    "QA automation lesson from real testing work",
    "Python coding habit for better testing",
    "API testing and automation quality",
    "Job search reflection for UK tech roles",
    "Interview preparation lesson",
    "AI tools and practical learning",
    "Machine learning project progress",
    "SDET to Data Science career transition",
    "Weekly reflection on learning and applications",
]


## Optional: preview coded PNG visual types

This notebook uses Python to create professional PNG graphics. It does not call Gemini, Imagen, OpenAI, Pexels, or any other image API.


In [ ]:
print('Available coded PNG visual types:')
for visual_type in CODED_VISUAL_TYPES:
    print('-', visual_type)


Available coded PNG visual types:
- data_dashboard
- qa_workflow
- api_testing_report
- python_code_snippet
- job_search_tracker
- learning_roadmap
- project_showcase
- achievement_tracker


## 4️⃣ Topic rotation (avoid repeats)

Keeps a small history file in Google Drive so the agent doesn't pick the same category twice in a week.

In [ ]:
from google.colab import drive
import json, os, random
from datetime import datetime, timedelta

drive.mount('/content/drive', force_remount=False)

HISTORY_DIR = '/content/drive/MyDrive/linkedin_agent'
HISTORY_FILE = f'{HISTORY_DIR}/topic_history.json'
os.makedirs(HISTORY_DIR, exist_ok=True)

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE) as f:
            return json.load(f)
    return []

def save_history(history):
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)

def pick_topic():
    history = load_history()
    cutoff = datetime.now() - timedelta(days=7)
    recent = {h['topic'] for h in history if datetime.fromisoformat(h['date']) > cutoff}
    available = [t for t in DAILY_TOPIC_BANK if t not in recent]
    if not available:
        available = DAILY_TOPIC_BANK  # all used recently → reset
    topic = random.choice(available)
    history.append({'date': datetime.now().isoformat(), 'topic': topic})
    save_history(history[-30:])  # keep last 30 only
    return topic

todays_topic = pick_topic()
print(f'🎯 Today\'s topic category: {todays_topic}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🎯 Today's topic category: Job search reflection for UK tech roles


## 5️⃣ Fetch fresh news (Hacker News + arXiv)

Grounds the post in something current so it doesn't feel generic.

In [ ]:
import requests, feedparser

def fetch_hackernews_top(n=5):
    """Top stories from HN, filtered for tech/AI relevance."""
    ids = requests.get('https://hacker-news.firebaseio.com/v0/topstories.json', timeout=10).json()[:30]
    stories = []
    keywords = ['ai', 'ml', 'llm', 'gpt', 'claude', 'python', 'data', 'model', 'neural', 'machine learning',
                'deep learning', 'transformer', 'agent', 'rag', 'embedding', 'open source', 'github']
    for sid in ids:
        try:
            s = requests.get(f'https://hacker-news.firebaseio.com/v0/item/{sid}.json', timeout=5).json()
            title = (s.get('title') or '').lower()
            if any(k in title for k in keywords):
                stories.append({
                    'title': s.get('title'),
                    'url': s.get('url', f'https://news.ycombinator.com/item?id={sid}'),
                    'score': s.get('score', 0)
                })
            if len(stories) >= n:
                break
        except Exception:
            continue
    return stories

def fetch_arxiv_recent(n=3):
    """Recent papers from arXiv cs.LG / cs.AI / cs.CL."""
    url = 'http://export.arxiv.org/api/query?search_query=cat:cs.LG+OR+cat:cs.AI+OR+cat:cs.CL&sortBy=submittedDate&sortOrder=descending&max_results=' + str(n)
    feed = feedparser.parse(url)
    return [{'title': e.title, 'summary': e.summary[:300], 'url': e.link} for e in feed.entries]

hn_stories = fetch_hackernews_top(5)
arxiv_papers = fetch_arxiv_recent(3)

print(f'📰 Pulled {len(hn_stories)} HN stories + {len(arxiv_papers)} arXiv papers')
for s in hn_stories[:3]:
    print(f'  • [{s["score"]}] {s["title"]}')

📰 Pulled 5 HN stories + 3 arXiv papers
  • [505] Local AI needs to be the norm
  • [52] Running local models on an M4 with 24GB memory
  • [5] Make America AI Ready: Strengths, Weaknesses, and Recommendations


## 6️⃣ Generate the post with Claude

In [ ]:

from anthropic import Anthropic

client = Anthropic()

# Format news context for the prompt
news_context = "## Today's tech/AI news from Hacker News:\n"
for s in hn_stories:
    news_context += f'- {s["title"]} ({s["url"]})\n'
news_context += "\n## Recent arXiv papers in AI/ML:\n"
for p in arxiv_papers:
    news_context += f'- {p["title"]}\n  Summary: {p["summary"][:200]}...\n'

system_prompt = f"""You are a ghostwriter crafting LinkedIn posts for {YOUR_NAME}, a {YOUR_ROLE}.

Audience: {YOUR_AUDIENCE}

Voice guidelines:
{YOUR_TONE}

LinkedIn best practices:
- Strong hook in the first 1 or 2 lines
- 150 to 280 words total
- Short paragraphs with white space
- End with a simple question or invitation for discussion
- 3 to 5 relevant hashtags at the end
- No em dash, no fancy symbols, no emojis, no special characters
- Do not use phrases like "In today's fast-paced world" or "Let's dive in"
- Make the writing sound natural and human, not AI-generated
"""

user_prompt = f"""Write today's LinkedIn post.

Topic category for today: {todays_topic}

Today's tech context. Use only if genuinely relevant. Do not force it:
{news_context}

Return only the final LinkedIn post text. Start directly with the hook."""

response = client.messages.create(
    model='claude-opus-4-7',
    max_tokens=1024,
    system=system_prompt,
    messages=[{'role': 'user', 'content': user_prompt}]
)

raw_draft = response.content[0].text.strip()
print('=' * 60)
print('RAW DRAFT POST')
print('=' * 60)
print(raw_draft)
print('=' * 60)
print(f'Raw word count: {len(raw_draft.split())}')


RAW DRAFT POST
Six months into job hunting in the UK, here is what nobody tells you.

Applying to 200 roles a week is not a strategy. It is a way to feel busy while getting nowhere.

What actually moved the needle for me was the opposite. Fewer applications. More research on the company. A tailored CV that matched the exact language in the job description. A short note explaining why I cared about that specific team.

The QA and SDET market in the UK right now is strange. Companies want Playwright, API testing, CI pipelines, cloud, and increasingly some exposure to AI testing or LLM evaluation. The bar keeps moving. But hiring managers are also tired of generic applications that could have been sent to anyone.

A few things that helped me:

Treating each application like a small project, not a lottery ticket.

Talking to people already in the team before applying when possible.

Being honest about what I do not know yet, instead of overselling.

Following up once, politely, then moving

## 7️⃣ Humanise text and create Claude-only coded PNG visuals

This section does two things:

1. Cleans the Claude draft using a humaniser function
2. Creates a professional LinkedIn PNG visual using Python code only

This version does **not** create realistic AI photos, quote cards, word-clouds, or plain text posters. It creates skill-based visuals such as dashboards, QA workflows, API testing reports, code snippet cards, job-search trackers, and learning roadmaps.


In [ ]:
import os, re, html, textwrap, unicodedata, random, math
from datetime import date
from PIL import Image, ImageDraw, ImageFont, ImageOps

OUTPUT_DIR = f'{HISTORY_DIR}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROJECT_SCREENSHOT_DIR, exist_ok=True)

DATE_STAMP = date.today().isoformat()
POST_CARD_PATH = f'{OUTPUT_DIR}/linkedin_post_card_{DATE_STAMP}.png'
VISUAL_IMAGE_PATH = f'{OUTPUT_DIR}/linkedin_visual_{DATE_STAMP}.png'

COLORS = {
    'bg': (248, 250, 252), 'card': (255, 255, 255), 'ink': (15, 23, 42),
    'muted': (71, 85, 105), 'line': (203, 213, 225), 'blue': (37, 99, 235),
    'green': (22, 163, 74), 'orange': (234, 88, 12), 'purple': (124, 58, 237),
    'red': (220, 38, 38), 'soft_blue': (219, 234, 254), 'soft_green': (220, 252, 231),
    'soft_orange': (255, 237, 213), 'soft_purple': (237, 233, 254), 'soft_red': (254, 226, 226),
}


def humanise_text(text):
    """Clean AI-looking text and remove/normalise special characters."""
    text = unicodedata.normalize('NFKC', text)
    replacements = {'—': '-', '–': '-', '“': '"', '”': '"', '‘': "'", '’': "'", '…': '...', '•': '-', '→': 'to', '✅': '', '🚀': '', '💡': '', '🔥': ''}
    for old, new in replacements.items():
        text = text.replace(old, new)
    if REMOVE_EMOJIS:
        text = re.sub(r'[\U00010000-\U0010ffff]', '', text)
    text = re.sub(r"[^\w\s\.,;:!?\-'\"/()#%&+@]", '', text, flags=re.UNICODE)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    hashtags = re.findall(r'#\w+', text)
    if len(hashtags) > MAX_HASHTAGS:
        allowed = set(hashtags[:MAX_HASHTAGS])
        text = ' '.join([w for w in text.split() if not w.startswith('#') or w in allowed])
    return text.strip()


def get_font(size=36, bold=False):
    possible_fonts = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf'
    ]
    for path in possible_fonts:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()


def draw_round_rect(draw, box, radius=24, fill='white', outline=None, width=1):
    draw.rounded_rectangle(box, radius=radius, fill=fill, outline=outline, width=width)


def text_size(draw, text, font):
    bbox = draw.textbbox((0, 0), text, font=font)
    return bbox[2] - bbox[0], bbox[3] - bbox[1]


def draw_wrapped_text(draw, text, xy, font, fill, max_width, line_gap=8, max_lines=None):
    x, y = xy
    lines = []
    for paragraph in text.split('\n'):
        words = paragraph.split()
        current = ''
        for word in words:
            test = (current + ' ' + word).strip()
            if text_size(draw, test, font)[0] <= max_width:
                current = test
            else:
                if current:
                    lines.append(current)
                current = word
        if current:
            lines.append(current)
        if not words:
            lines.append('')
    if max_lines and len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip('.') + '...'
    for line in lines:
        draw.text((x, y), line, fill=fill, font=font)
        y += font.size + line_gap
    return y


def base_canvas(title, subtitle, visual_type):
    w, h = 1200, 628
    img = Image.new('RGB', (w, h), COLORS['bg'])
    draw = ImageDraw.Draw(img)
    for x in range(0, w, 60):
        draw.line((x, 0, x, h), fill=(241, 245, 249), width=1)
    for y in range(0, h, 60):
        draw.line((0, y, w, y), fill=(241, 245, 249), width=1)
    title_font, subtitle_font, small_font = get_font(44, True), get_font(24), get_font(18)
    draw.text((52, 34), title, fill=COLORS['ink'], font=title_font)
    draw_wrapped_text(draw, subtitle, (54, 92), subtitle_font, COLORS['muted'], 690, line_gap=5, max_lines=2)
    pill = visual_type.replace('_', ' ').title()
    pill_w = text_size(draw, pill, small_font)[0] + 36
    draw_round_rect(draw, (1200 - pill_w - 52, 42, 1200 - 52, 82), 20, fill=COLORS['soft_blue'])
    draw.text((1200 - pill_w - 34, 52), pill, fill=COLORS['blue'], font=small_font)
    draw.text((54, 590), f'{YOUR_NAME} | {DATE_STAMP}', fill=COLORS['muted'], font=small_font)
    return img, draw


def detect_visual_category(topic, post_text):
    text = f"{topic} {post_text}".lower()
    if any(word in text for word in ['graduation', 'passed', 'completed', 'certificate', 'achievement', 'milestone', 'open to work']): return 'achievement_tracker'
    if any(word in text for word in ['interview', 'recruiter', 'job', 'cv', 'application', 'hiring', 'linkedin']): return 'job_search_tracker'
    if any(word in text for word in ['api', 'endpoint', 'postman', 'status code', 'response time']): return 'api_testing_report'
    if any(word in text for word in ['qa', 'testing', 'selenium', 'playwright', 'pytest', 'automation', 'sdet', 'test report']): return 'qa_workflow'
    if any(word in text for word in ['project', 'dashboard', 'model', 'accuracy', 'automation framework', 'ml project']): return 'project_showcase'
    if any(word in text for word in ['data science', 'machine learning', 'ai', 'analytics', 'visualization', 'visualisation', 'llm', 'mlops']): return 'data_dashboard'
    if any(word in text for word in ['learning', 'lesson', 'studying', 'course', 'roadmap']): return 'learning_roadmap'
    return 'python_code_snippet'


def create_linkedin_post_card(post_text, topic, output_path):
    img, draw = base_canvas('LinkedIn Draft', topic, 'text_card')
    body_font = get_font(24)
    draw_round_rect(draw, (52, 150, 1148, 548), 28, fill=COLORS['card'], outline=COLORS['line'])
    draw_wrapped_text(draw, post_text, (90, 184), body_font, COLORS['ink'], 1020, line_gap=7, max_lines=12)
    img.save(output_path, quality=95)
    return output_path


def draw_kpi_card(draw, x, y, w, h, label, value, accent):
    draw_round_rect(draw, (x, y, x+w, y+h), 22, fill=COLORS['card'], outline=COLORS['line'])
    draw.text((x+22, y+18), label, fill=COLORS['muted'], font=get_font(18))
    draw.text((x+22, y+48), value, fill=COLORS['ink'], font=get_font(34, True))
    draw.rounded_rectangle((x+22, y+h-18, x+w-22, y+h-10), radius=5, fill=COLORS['line'])
    draw.rounded_rectangle((x+22, y+h-18, x+int(w*0.72), y+h-10), radius=5, fill=accent)


def create_data_dashboard(output_path, topic, post_text):
    img, draw = base_canvas('Data Science Dashboard', 'Python, analytics, ML thinking and business-ready insights', 'data_dashboard')
    draw_kpi_card(draw, 54, 160, 235, 118, 'Data quality', '94%', COLORS['green'])
    draw_kpi_card(draw, 314, 160, 235, 118, 'Model check', '0.87', COLORS['blue'])
    draw_kpi_card(draw, 574, 160, 235, 118, 'Features', '18', COLORS['purple'])
    draw_kpi_card(draw, 834, 160, 235, 118, 'Next action', 'Test', COLORS['orange'])
    draw_round_rect(draw, (54, 310, 690, 552), 28, fill=COLORS['card'], outline=COLORS['line'])
    draw.text((82, 334), 'Learning signal over time', fill=COLORS['ink'], font=get_font(24, True))
    chart = [(105, 500), (190, 455), (275, 470), (360, 405), (445, 422), (530, 365), (615, 338)]
    for i in range(5):
        y = 500 - i*36
        draw.line((95, y, 650, y), fill=(226, 232, 240), width=1)
    draw.line(chart, fill=COLORS['blue'], width=5)
    for p in chart:
        draw.ellipse((p[0]-7, p[1]-7, p[0]+7, p[1]+7), fill=COLORS['blue'])
    draw_round_rect(draw, (730, 310, 1148, 552), 28, fill=COLORS['card'], outline=COLORS['line'])
    draw.text((760, 334), 'Tech stack focus', fill=COLORS['ink'], font=get_font(24, True))
    items = [('Python', COLORS['soft_blue'], COLORS['blue']), ('SQL', COLORS['soft_green'], COLORS['green']), ('Pandas', COLORS['soft_purple'], COLORS['purple']), ('Model evaluation', COLORS['soft_orange'], COLORS['orange'])]
    y = 382
    for label, bg, fg in items:
        draw_round_rect(draw, (760, y, 1098, y+44), 16, fill=bg)
        draw.text((782, y+11), label, fill=fg, font=get_font(20, True))
        y += 52
    img.save(output_path, quality=95); return output_path


def create_qa_workflow(output_path, topic, post_text):
    img, draw = base_canvas('QA Automation Workflow', 'From requirement to reliable release', 'qa_workflow')
    steps = [('Requirement', 'Understand risk'), ('Test cases', 'Cover key paths'), ('Automation', 'Pytest / API'), ('CI/CD', 'Run on commit'), ('Report', 'Fix fast')]
    x, y = 70, 230
    for i, (title, sub) in enumerate(steps):
        draw_round_rect(draw, (x, y, x+190, y+120), 26, fill=COLORS['card'], outline=COLORS['line'])
        draw.text((x+24, y+28), title, fill=COLORS['ink'], font=get_font(23, True))
        draw.text((x+24, y+68), sub, fill=COLORS['muted'], font=get_font(18))
        if i < len(steps)-1:
            draw.line((x+190, y+60, x+230, y+60), fill=COLORS['blue'], width=4)
            draw.polygon([(x+230, y+60), (x+214, y+50), (x+214, y+70)], fill=COLORS['blue'])
        x += 225
    draw_round_rect(draw, (90, 410, 1110, 540), 28, fill=COLORS['soft_blue'])
    draw.text((125, 438), 'Professional testing mindset', fill=COLORS['blue'], font=get_font(28, True))
    draw.text((125, 482), 'Automate repeated checks. Use human judgement for risk, edge cases, and product quality.', fill=COLORS['ink'], font=get_font(24))
    img.save(output_path, quality=95); return output_path


def create_api_testing_report(output_path, topic, post_text):
    img, draw = base_canvas('API Testing Report', 'A clean testing snapshot for backend quality', 'api_testing_report')
    draw_round_rect(draw, (66, 150, 480, 550), 28, fill=COLORS['card'], outline=COLORS['line'])
    draw.text((96, 182), 'Test summary', fill=COLORS['ink'], font=get_font(30, True))
    metrics = [('Total tests', '128', COLORS['blue']), ('Passed', '119', COLORS['green']), ('Failed', '6', COLORS['red']), ('Skipped', '3', COLORS['orange'])]
    y = 238
    for label, val, color in metrics:
        draw.text((100, y), label, fill=COLORS['muted'], font=get_font(22)); draw.text((360, y-4), val, fill=color, font=get_font(28, True)); y += 64
    draw_round_rect(draw, (545, 150, 1135, 550), 28, fill=COLORS['card'], outline=COLORS['line'])
    draw.text((580, 182), 'Endpoint checks', fill=COLORS['ink'], font=get_font(30, True))
    endpoints = [('/auth/login', '200 OK', COLORS['green']), ('/users/{id}', '200 OK', COLORS['green']), ('/orders', '201 Created', COLORS['green']), ('/payment', '422 Validated', COLORS['orange']), ('/reports', '500 Needs fix', COLORS['red'])]
    y = 238
    for endpoint, status, color in endpoints:
        draw_round_rect(draw, (580, y, 1100, y+44), 14, fill=COLORS['bg'])
        draw.text((600, y+11), endpoint, fill=COLORS['ink'], font=get_font(19)); draw.text((880, y+11), status, fill=color, font=get_font(19, True)); y += 58
    img.save(output_path, quality=95); return output_path


def create_python_code_snippet(output_path, topic, post_text):
    img, draw = base_canvas('Python Testing Snippet', 'Small habits that improve automation quality', 'python_code_snippet')
    draw_round_rect(draw, (70, 150, 1130, 545), 28, fill=(15, 23, 42), outline=COLORS['line'])
    code_font, header_font = get_font(24), get_font(20, True)
    draw.text((105, 180), 'test_api_response.py', fill=(226, 232, 240), font=header_font)
    code = ['import requests', '', 'def test_user_api_returns_success():', '    response = requests.get(BASE_URL + "/users/1")', '    assert response.status_code == 200', '    assert response.elapsed.total_seconds() < 1.0', '    assert "id" in response.json()', '', '# Reliable tests are simple, clear, and repeatable.']
    y = 230
    for i, line in enumerate(code, 1):
        draw.text((105, y), f'{i:02}', fill=(100, 116, 139), font=code_font); draw.text((155, y), line, fill=(226, 232, 240), font=code_font); y += 34
    img.save(output_path, quality=95); return output_path


def create_job_search_tracker(output_path, topic, post_text):
    img, draw = base_canvas('UK Tech Job Search Tracker', 'A practical weekly view: applications, learning, interviews, improvements', 'job_search_tracker')
    cards = [('Applications', 18, COLORS['blue']), ('CV versions', 4, COLORS['purple']), ('Interviews', 2, COLORS['green']), ('Skills improved', 3, COLORS['orange'])]
    x = 70
    for label, value, color in cards:
        draw_kpi_card(draw, x, 160, 245, 120, label, str(value), color); x += 275
    draw_round_rect(draw, (70, 325, 690, 545), 28, fill=COLORS['card'], outline=COLORS['line'])
    draw.text((105, 355), 'This week focus', fill=COLORS['ink'], font=get_font(28, True))
    focus = ['Tailor CV for SDET roles', 'Show Python/API testing projects', 'Practise interview examples', 'Track sponsor-friendly companies']
    y = 405
    for item in focus:
        draw.ellipse((110, y+7, 126, y+23), fill=COLORS['green']); draw.text((145, y), item, fill=COLORS['ink'], font=get_font(22)); y += 42
    draw_round_rect(draw, (730, 325, 1130, 545), 28, fill=COLORS['soft_blue'])
    draw.text((765, 355), 'Recruiter signal', fill=COLORS['blue'], font=get_font(28, True))
    draw_wrapped_text(draw, 'Clear profile + real projects + focused applications beats random applying.', (765, 405), get_font(26), COLORS['ink'], 320, line_gap=10, max_lines=4)
    img.save(output_path, quality=95); return output_path


def create_learning_roadmap(output_path, topic, post_text):
    img, draw = base_canvas('Learning Roadmap', 'Turning daily practice into visible LinkedIn progress', 'learning_roadmap')
    months = [('Week 1', 'Python basics'), ('Week 2', 'API testing'), ('Week 3', 'Data project'), ('Week 4', 'LinkedIn proof')]
    x0, y0 = 100, 245
    for i, (week, item) in enumerate(months):
        x = x0 + i*260
        draw.ellipse((x, y0, x+72, y0+72), fill=COLORS['blue']); draw.text((x+20, y0+22), str(i+1), fill='white', font=get_font(26, True))
        draw.text((x-10, y0+95), week, fill=COLORS['ink'], font=get_font(24, True)); draw_wrapped_text(draw, item, (x-30, y0+132), get_font(20), COLORS['muted'], 170, max_lines=2)
        if i < 3:
            draw.line((x+72, y0+36, x+250, y0+36), fill=COLORS['line'], width=5); draw.polygon([(x+250, y0+36), (x+234, y0+26), (x+234, y0+46)], fill=COLORS['line'])
    draw_round_rect(draw, (140, 470, 1060, 540), 24, fill=COLORS['soft_green'])
    draw.text((175, 492), 'Goal: make learning visible through projects, screenshots, and simple explanations.', fill=COLORS['green'], font=get_font(24, True))
    img.save(output_path, quality=95); return output_path


def create_project_showcase(output_path, topic, post_text):
    valid_ext = ('.png', '.jpg', '.jpeg')
    files = [f for f in os.listdir(PROJECT_SCREENSHOT_DIR) if f.lower().endswith(valid_ext)] if os.path.exists(PROJECT_SCREENSHOT_DIR) else []
    if files:
        selected = os.path.join(PROJECT_SCREENSHOT_DIR, random.choice(files))
        try:
            screenshot = Image.open(selected).convert('RGB')
            screenshot = ImageOps.fit(screenshot, (1200, 628), method=Image.Resampling.LANCZOS)
            screenshot.save(output_path, quality=95)
            print(f'Using real project screenshot: {selected}')
            return output_path
        except Exception as e:
            print(f'Could not load screenshot. Creating coded project visual instead: {e}')
    img, draw = base_canvas('Project Showcase', 'A clean way to show real skills without overexplaining', 'project_showcase')
    draw_round_rect(draw, (70, 150, 760, 545), 28, fill=COLORS['card'], outline=COLORS['line'])
    draw.text((105, 182), 'Project pipeline', fill=COLORS['ink'], font=get_font(30, True))
    pipeline = [('Data', COLORS['blue']), ('Clean', COLORS['purple']), ('Model', COLORS['orange']), ('Test', COLORS['green']), ('Share', COLORS['red'])]
    x = 115
    for label, color in pipeline:
        draw_round_rect(draw, (x, 270, x+105, 330), 18, fill=color); draw.text((x+20, 288), label, fill='white', font=get_font(20, True))
        if label != 'Share': draw.line((x+105, 300, x+140, 300), fill=COLORS['line'], width=4)
        x += 140
    draw_wrapped_text(draw, 'Show the problem, process, result, and one lesson learned. Recruiters understand proof faster than long claims.', (105, 390), get_font(24), COLORS['muted'], 590, line_gap=9, max_lines=4)
    draw_round_rect(draw, (800, 150, 1130, 545), 28, fill=COLORS['soft_purple'])
    draw.text((835, 190), 'What to include', fill=COLORS['purple'], font=get_font(28, True))
    y = 245
    for item in ['GitHub link', 'Dashboard screenshot', 'Testing proof', 'Short result']:
        draw.text((850, y), item, fill=COLORS['ink'], font=get_font(23, True)); y += 58
    img.save(output_path, quality=95); return output_path


def create_achievement_tracker(output_path, topic, post_text):
    img, draw = base_canvas('Achievement Tracker', 'Milestones, proof, and the next professional step', 'achievement_tracker')
    draw_round_rect(draw, (80, 165, 1120, 530), 34, fill=COLORS['card'], outline=COLORS['line'])
    items = [('Completed', 'MSc Data Science', COLORS['green']), ('Built', 'QA + Python experience', COLORS['blue']), ('Next', 'SDET / Data roles', COLORS['purple'])]
    x = 135
    for label, value, color in items:
        draw.ellipse((x, 220, x+86, 306), fill=color); draw.text((x+26, 246), '✓', fill='white', font=get_font(34, True))
        draw.text((x-15, 330), label, fill=color, font=get_font(25, True)); draw_wrapped_text(draw, value, (x-35, 370), get_font(24), COLORS['ink'], 220, max_lines=2)
        x += 350
    draw.text((150, 485), 'Use achievements as evidence, not just announcements.', fill=COLORS['muted'], font=get_font(26))
    img.save(output_path, quality=95); return output_path


VISUAL_BUILDERS = {
    'data_dashboard': create_data_dashboard, 'qa_workflow': create_qa_workflow, 'api_testing_report': create_api_testing_report,
    'python_code_snippet': create_python_code_snippet, 'job_search_tracker': create_job_search_tracker,
    'learning_roadmap': create_learning_roadmap, 'project_showcase': create_project_showcase, 'achievement_tracker': create_achievement_tracker,
}


def create_topic_visual(topic, post_text, output_path):
    if not USE_CODED_PNG_VISUALS:
        raise RuntimeError('USE_CODED_PNG_VISUALS is False. Turn it on to create Python PNG visuals.')
    category = detect_visual_category(topic, post_text)
    print(f'Detected coded visual category: {category}')
    builder = VISUAL_BUILDERS.get(category, create_python_code_snippet)
    return builder(output_path, topic, post_text)


draft = humanise_text(raw_draft)
post_card_path = None
if ATTACH_TEXT_POST_CARD:
    post_card_path = create_linkedin_post_card(draft, todays_topic, POST_CARD_PATH)
visual_image_path = create_topic_visual(todays_topic, draft, VISUAL_IMAGE_PATH)

print('=' * 60)
print('HUMANISED FINAL POST')
print('=' * 60)
print(draft)
print('=' * 60)
print(f'Final word count: {len(draft.split())}')
print(f'Text post card saved: {post_card_path if post_card_path else "disabled"}')
print(f'Claude-only coded PNG visual saved: {visual_image_path}')


Detected coded visual category: job_search_tracker
HUMANISED FINAL POST
Six months into job hunting in the UK, here is what nobody tells you.

Applying to 200 roles a week is not a strategy. It is a way to feel busy while getting nowhere.

What actually moved the needle for me was the opposite. Fewer applications. More research on the company. A tailored CV that matched the exact language in the job description. A short note explaining why I cared about that specific team.

The QA and SDET market in the UK right now is strange. Companies want Playwright, API testing, CI pipelines, cloud, and increasingly some exposure to AI testing or LLM evaluation. The bar keeps moving. But hiring managers are also tired of generic applications that could have been sent to anyone.

A few things that helped me:

Treating each application like a small project, not a lottery ticket.

Talking to people already in the team before applying when possible.

Being honest about what I do not know yet, instead 

## 8️⃣ Email the draft and visuals to yourself


In [ ]:
import smtplib, mimetypes
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from datetime import date
import os

def attach_file(msg, file_path):
    if not file_path or not os.path.exists(file_path):
        return
    ctype, encoding = mimetypes.guess_type(file_path)
    if ctype is None or encoding is not None:
        ctype = 'application/octet-stream'
    maintype, subtype = ctype.split('/', 1)
    with open(file_path, 'rb') as f:
        part = MIMEBase(maintype, subtype)
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', 'attachment', filename=os.path.basename(file_path))
    msg.attach(part)


def send_email(subject, body, html_body=None, attachments=None):
    msg = MIMEMultipart('mixed')
    msg['Subject'] = subject
    msg['From'] = GMAIL_ADDRESS
    msg['To'] = RECIPIENT_EMAIL
    alt = MIMEMultipart('alternative')
    alt.attach(MIMEText(body, 'plain'))
    if html_body:
        alt.attach(MIMEText(html_body, 'html'))
    msg.attach(alt)
    for file_path in attachments or []:
        attach_file(msg, file_path)
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
        server.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        server.send_message(msg)

safe_draft_html = html.escape(draft).replace('\n', '<br>')
attachments = [visual_image_path]
if post_card_path:
    attachments.append(post_card_path)
html_body = f"""
<html><body style='font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; max-width: 720px;'>
  <h2>Your LinkedIn draft for {date.today().strftime('%A, %b %d')}</h2>
  <p><b>Topic:</b> {html.escape(todays_topic)}</p>
  <hr>
  <div style='background:#f6f8fa; padding:20px; border-radius:8px; font-size:15px; line-height:1.6;'>
    {safe_draft_html}
  </div>
  <hr>
  <p><b>Attached:</b> Claude-only coded PNG visual.</p>
  <p style='color:#666; font-size:13px;'>Word count: {len(draft.split())} | Claude for text, Python for PNG visual</p>
</body></html>
"""
plain = f"""Your LinkedIn draft for {date.today()}

Topic: {todays_topic}

{'='*60}

{draft}

{'='*60}

Attached visual:
- Claude-only coded PNG visual: {visual_image_path}
"""
send_email(
    subject=f'LinkedIn draft with Claude-only PNG visual - {date.today().strftime("%b %d")}',
    body=plain,
    html_body=html_body,
    attachments=attachments
)
print(f'Sent draft and Claude-only coded PNG visual to {RECIPIENT_EMAIL}')


Sent draft and Claude-only coded PNG visual to mohithsangeetham@gmail.com


## Next steps

Run all cells from top to bottom.

This version uses:

1. `ANTHROPIC_API_KEY` for Claude text generation.
2. Python/PIL code for professional PNG visuals.
3. No Gemini, Imagen, OpenAI, Pexels, or paid image-generation API.
4. Keep `ATTACH_TEXT_POST_CARD = False` to avoid full text/quote-card images.
5. Optional: add real project screenshots into:
   `/content/drive/MyDrive/linkedin_agent/project_screenshots`

The email will include:
- The humanised LinkedIn draft created with Claude
- One professional coded PNG visual created by Python

To automate daily posting, schedule this Colab notebook or run it manually each day and review before posting.
